# 聚合物结构与电荷生成（修复版）

本 notebook 只使用 `mapped_charges.json` 映射电荷，不再使用 `charges[atom_idx]`。写出 `.chg` 前会校验原子数、元素顺序和总电荷。

In [ ]:
# -*- coding: utf-8 -*-
"""
基于 RESP 映射电荷生成随机共聚物结构与 .chg 文件。

功能目的：
    读取 mapped_charges.json，把重复单元按随机序列拼接成线性共聚物，并生成
    与最终 XYZ 原子顺序一致的 .chg 文件。

输入参数：
    主要入口为 build_random_copolymer(...)，输入为重复单元 SMILES、重复次数、
    mapped charge 文件路径、共聚物名称和随机种子。

返回值：
    PolymerBuildResult，包含 node_list、元素列表、电荷列表、XYZ 路径、CHG 路径。

关键流程：
    1. 根据 random_seed 和重复次数生成重复单元序列；
    2. 对每个重复单元使用原始含“*”的 AddHs Mol 生成节点、键和电荷；
    3. 删除内部“*”节点，只保留整条链首端和末端“*”；
    4. 根据 dummy 邻接原子顺序创建重复单元之间的单键；
    5. 从 node_list 和键列表构建 RDKit Mol，并把首末端“*”原位替换成 H；
    6. 生成 XYZ，并确认 XYZ 元素顺序与 node_list 转换后的元素顺序完全一致；
    7. 校验总电荷后写出 .chg。

可能报错或边界情况：
    - mapped charge 缺少节点、电荷总和不符、元素顺序不一致、RDKit 构象失败时
      会抛出异常并阻止写出错误 .chg。
"""

from __future__ import annotations

import json
import random
import re
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Sequence, Tuple

import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdchem import RWMol


FLOAT_TOL = 1.0e-4


@dataclass
class PolymerBuildResult:
    """聚合物构建结果。"""

    copolymer_name: str
    sequence: List[str]
    cyclic: bool
    expected_charge: float
    actual_charge: float
    node_list: List[str]
    xyz_elements: List[str]
    charge_list: List[float]
    xyz_path: Path
    chg_path: Path
    node_list_path: Path


BOND_LIST = [
    Chem.rdchem.BondType.UNSPECIFIED,
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.QUADRUPLE,
    Chem.rdchem.BondType.QUINTUPLE,
    Chem.rdchem.BondType.HEXTUPLE,
    Chem.rdchem.BondType.ONEANDAHALF,
    Chem.rdchem.BondType.TWOANDAHALF,
    Chem.rdchem.BondType.THREEANDAHALF,
    Chem.rdchem.BondType.FOURANDAHALF,
    Chem.rdchem.BondType.FIVEANDAHALF,
    Chem.rdchem.BondType.AROMATIC,
    Chem.rdchem.BondType.IONIC,
    Chem.rdchem.BondType.HYDROGEN,
    Chem.rdchem.BondType.THREECENTER,
    Chem.rdchem.BondType.DATIVEONE,
    Chem.rdchem.BondType.DATIVE,
    Chem.rdchem.BondType.DATIVEL,
    Chem.rdchem.BondType.DATIVER,
    Chem.rdchem.BondType.OTHER,
    Chem.rdchem.BondType.ZERO,
]


def safe_name(name: str) -> str:
    """功能目的：把名称转换为安全文件名。"""
    return "".join(ch if ch.isalnum() or ch in "._+-" else "_" for ch in str(name))


def load_mapped_charges(path: Path) -> Dict[str, Any]:
    """
    功能目的：
        读取单个重复单元的 mapped_charges.json。

    输入参数：
        path: JSON 文件路径。

    返回值：
        JSON 字典。

    可能报错或边界情况：
        文件不存在或格式不正确时由标准库抛出异常。
    """
    with Path(path).open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    if "mapped_charges" not in payload:
        raise ValueError(f"{path} 缺少 mapped_charges 字段")
    return payload


def build_sequence(repeating_unit: Dict[str, int], random_seed: int) -> List[str]:
    """
    功能目的：
        按旧 notebook 的随机逻辑生成共聚物重复单元序列。

    输入参数：
        repeating_unit: 名称 -> 出现次数。
        random_seed: 随机种子。

    返回值：
        重复单元名称序列。

    关键流程：
        使用 random.choice 在剩余计数中抽取，保证与旧 notebook 相同种子下结果一致。

    可能报错或边界情况：
        出现次数小于 0 或总数为 0 时抛出 ValueError。
    """
    counts = {name: int(count) for name, count in repeating_unit.items()}
    if any(count < 0 for count in counts.values()) or sum(counts.values()) <= 0:
        raise ValueError(f"重复单元数量非法: {repeating_unit}")

    random.seed(int(random_seed))
    names = list(counts.keys())
    tracker = {name: 0 for name in names}
    sequence: List[str] = []
    for _ in range(sum(counts.values())):
        while True:
            name = random.choice(names)
            if tracker[name] < counts[name]:
                tracker[name] += 1
                sequence.append(name)
                break
    return sequence


def build_block_sequence(repeating_unit: Dict[str, int], number_of_blocks: int) -> List[str]:
    """
    功能目的：
        按旧 block copolymer notebook 的规则生成嵌段共聚物序列。

    输入参数：
        repeating_unit: 名称 -> 每个 block 内的重复次数。
        number_of_blocks: block 数量。

    返回值：
        重复单元名称序列，例如 A A B B A A B B。

    关键流程：
        保留 Excel 中聚合物行的顺序，每个 block 依次展开各单体。

    可能报错或边界情况：
        block 数或重复次数非法时抛出 ValueError。
    """
    n_blocks = int(number_of_blocks)
    if n_blocks <= 0:
        raise ValueError(f"Number of blocks 必须大于 0，实际为 {number_of_blocks}")
    counts = {name: int(count) for name, count in repeating_unit.items()}
    if any(count < 0 for count in counts.values()) or sum(counts.values()) <= 0:
        raise ValueError(f"重复单元数量非法: {repeating_unit}")

    sequence: List[str] = []
    for _ in range(n_blocks):
        for name, repeats in counts.items():
            sequence.extend([name] * repeats)
    return sequence


def build_homopolymer_sequence(name: str, repeating_unit: int) -> List[str]:
    """
    功能目的：
        生成均聚物序列。

    输入参数：
        name: 单体名称。
        repeating_unit: 聚合度。

    返回值：
        长度为 repeating_unit 的名称列表。

    可能报错或边界情况：
        聚合度小于等于 0 时抛出 ValueError。
    """
    n_repeat = int(repeating_unit)
    if n_repeat <= 0:
        raise ValueError(f"{name}: repeating unit 必须大于 0，实际为 {repeating_unit}")
    return [str(name)] * n_repeat


def node_label(atom: Chem.Atom, unit_name: str, repeat_index1: int) -> str:
    """
    功能目的：
        按旧流程生成聚合物节点名称。

    输入参数：
        atom: RDKit Atom。
        unit_name: 重复单元名称。
        repeat_index1: 聚合链中第几个重复单元，1-based。

    返回值：
        节点名称，例如 O1_CMC_4 或 *8_celllulose_7。
    """
    symbol = atom.GetSymbol()
    idx = atom.GetIdx()
    prefix = "*" if symbol == "*" else symbol
    return f"{prefix}{idx}_{unit_name}_{repeat_index1}"


def base_node_label(atom: Chem.Atom, unit_name: str) -> str:
    """
    功能目的：
        生成与 mapped_charges.json 对应的不含重复序号的节点名。
    """
    symbol = atom.GetSymbol()
    idx = atom.GetIdx()
    prefix = "*" if symbol == "*" else symbol
    return f"{prefix}{idx}_{unit_name}"


def node_to_element(node: str) -> str:
    """
    功能目的：
        从节点名提取元素符号。

    输入参数：
        node: 节点名。

    返回值：
        元素符号；若节点以“*”开头返回“*”。

    可能报错或边界情况：
        节点无法解析时抛出 ValueError。
    """
    if node.startswith("*"):
        return "*"
    match = re.match(r"([A-Z][a-z]*)\d+_", node)
    if not match:
        raise ValueError(f"无法从节点名解析元素: {node}")
    return match.group(1)


def capped_node_element(node: str) -> str:
    """
    功能目的：
        返回最终封端后 XYZ 中应出现的元素。
    """
    return "H" if node.startswith("*") else node_to_element(node)


def formal_charge_for_node(node: str, mapped_payloads: Dict[str, Dict[str, Any]]) -> int:
    """
    功能目的：
        根据节点名查找该原子的形式电荷。

    输入参数：
        node: 带重复序号的节点名。
        mapped_payloads: 重复单元 mapped_charges payload。

    返回值：
        整数形式电荷。
    """
    parts = node.rsplit("_", 1)
    if len(parts) != 2:
        return 0
    base = parts[0]
    unit_name = base.rsplit("_", 1)[-1]
    payload = mapped_payloads.get(unit_name)
    if payload is None:
        return 0
    entry = payload["mapped_charges"].get(base)
    if entry is None:
        return 0
    return int(entry.get("formal_charge", 0))


def collect_polymer_graph(
    smiles_by_name: Dict[str, str],
    mapped_payloads: Dict[str, Dict[str, Any]],
    sequence: Sequence[str],
    cyclic: bool = False,
) -> Tuple[Dict[str, float], Dict[str, int], List[Tuple[str, str]], List[Tuple[Tuple[str, str], int]]]:
    """
    功能目的：
        生成聚合物节点电荷、形式电荷和连接关系。

    输入参数：
        smiles_by_name: 重复单元名称 -> SMILES。
        mapped_payloads: 重复单元 mapped_charges。
        sequence: 聚合链重复单元序列。

    返回值：
        (final_charge_dict, formal_charge_dict, final_bonds, final_bonds_with_order)。

    关键流程：
        先收集所有重复单元内部节点和键，再把所有 dummy 键按链顺序处理：
        - 线性链：内部 dummy 两两拼接，首末 dummy 保留为封端 H；
        - 环状链：内部 dummy 两两拼接，首末 dummy 的邻接非 dummy 原子直接闭环。

    可能报错或边界情况：
        缺少 mapped charge、重复单元没有恰好两个 dummy 键时抛出异常。
    """
    all_charges: Dict[str, float] = {}
    formal_charges: Dict[str, int] = {}
    all_bonds: List[Tuple[str, str]] = []
    all_bonds_with_order: List[Tuple[Tuple[str, str], int]] = []
    dummy_bonds: List[Tuple[str, str]] = []

    for repeat_idx0, unit_name in enumerate(sequence):
        repeat_idx1 = repeat_idx0 + 1
        mol = Chem.MolFromSmiles(smiles_by_name[unit_name])
        if mol is None:
            raise ValueError(f"{unit_name}: SMILES 无法解析")
        mol_with_h = Chem.AddHs(mol)
        mapped = mapped_payloads[unit_name]["mapped_charges"]

        unit_dummy_bonds: List[Tuple[str, str]] = []
        for atom in mol_with_h.GetAtoms():
            base_label = base_node_label(atom, unit_name)
            if base_label not in mapped:
                raise KeyError(f"{unit_name}: mapped charge 缺少节点 {base_label}")
            full_label = node_label(atom, unit_name, repeat_idx1)
            all_charges[full_label] = float(mapped[base_label]["charge"])
            q = int(mapped[base_label].get("formal_charge", atom.GetFormalCharge()))
            if q != 0:
                formal_charges[full_label] = q

        for bond in mol_with_h.GetBonds():
            begin = mol_with_h.GetAtomWithIdx(bond.GetBeginAtomIdx())
            end = mol_with_h.GetAtomWithIdx(bond.GetEndAtomIdx())
            begin_node = node_label(begin, unit_name, repeat_idx1)
            end_node = node_label(end, unit_name, repeat_idx1)
            order = BOND_LIST.index(bond.GetBondType())
            all_bonds.append((begin_node, end_node))
            all_bonds_with_order.append(((begin_node, end_node), order))
            if begin.GetSymbol() == "*" or end.GetSymbol() == "*":
                unit_dummy_bonds.append((begin_node, end_node))

        if len(unit_dummy_bonds) != 2:
            raise ValueError(f"{unit_name}_{repeat_idx1}: 需要两个 dummy 键，实际 {len(unit_dummy_bonds)}")
        dummy_bonds.extend(unit_dummy_bonds)

    non_virtual_atoms: List[str] = []
    for bond in dummy_bonds:
        left, right = bond
        if left.startswith("*"):
            non_virtual_atoms.append(right)
        else:
            non_virtual_atoms.append(left)

    inter_bonds: List[Tuple[str, str]] = []
    for idx in range(1, len(non_virtual_atoms) - 1, 2):
        inter_bonds.append((non_virtual_atoms[idx], non_virtual_atoms[idx + 1]))

    if cyclic:
        # 中文注释：环状聚合物不保留任何 dummy 节点；首末两个聚合位点邻接的真实原子直接成键闭环。
        inter_bonds.append((non_virtual_atoms[0], non_virtual_atoms[-1]))
        final_charges = {
            key: value
            for key, value in all_charges.items()
            if not key.startswith("*")
        }
    else:
        # 中文注释：线性聚合物只保留整条链首末两个 dummy，后续原位替换为 H 并沿用封端 H 的 RESP 电荷。
        end_dummy_charge_dict: Dict[str, float] = {}
        first_dummy_key: str | None = None
        last_dummy_key: str | None = None
        for atom_name in all_charges:
            if atom_name.startswith("*"):
                if first_dummy_key is None:
                    first_dummy_key = atom_name
                last_dummy_key = atom_name
        if first_dummy_key is None or last_dummy_key is None:
            raise ValueError("聚合物链没有找到首末端 dummy 原子")
        end_dummy_charge_dict[first_dummy_key] = all_charges[first_dummy_key]
        end_dummy_charge_dict[last_dummy_key] = all_charges[last_dummy_key]

        final_charges = {
            key: value
            for key, value in all_charges.items()
            if not key.startswith("*") or key in end_dummy_charge_dict
        }
        inter_bonds.insert(0, dummy_bonds[0])
        inter_bonds.append(dummy_bonds[-1])

    inter_bonds_with_order = [(bond, 1) for bond in inter_bonds]

    intra_bonds = [bond for bond in all_bonds if not bond[0].startswith("*") and not bond[1].startswith("*")]
    intra_bonds_with_order = [
        item
        for item in all_bonds_with_order
        if not item[0][0].startswith("*") and not item[0][1].startswith("*")
    ]

    final_bonds = intra_bonds + inter_bonds
    final_bonds_with_order = intra_bonds_with_order + inter_bonds_with_order
    return final_charges, formal_charges, final_bonds, final_bonds_with_order


def graph_to_mol(
    node_list: Sequence[str],
    bonds_with_order: Sequence[Tuple[Tuple[str, str], int]],
    formal_charges: Dict[str, int],
) -> Chem.Mol:
    """
    功能目的：
        将节点列表和键级关系转换为 RDKit Mol。

    输入参数：
        node_list: 节点顺序，必须与电荷列表一致。
        bonds_with_order: ((node1, node2), bond_order_index) 列表。
        formal_charges: 节点名 -> 形式电荷。

    返回值：
        RDKit Mol。

    关键流程：
        按 node_list 顺序添加原子，再根据键级添加键，最后设置形式电荷并 sanitize。

    可能报错或边界情况：
        键引用不存在节点或 RDKit sanitize 失败时抛出异常。
    """
    rw_mol = Chem.RWMol()
    node_to_idx: Dict[str, int] = {}
    for node in node_list:
        atom = Chem.Atom(node_to_element(node))
        atom_idx = rw_mol.AddAtom(atom)
        node_to_idx[node] = atom_idx

    for (node1, node2), order_idx in bonds_with_order:
        if node1 not in node_to_idx or node2 not in node_to_idx:
            raise KeyError(f"键引用了不存在的节点: {node1}, {node2}")
        rw_mol.AddBond(node_to_idx[node1], node_to_idx[node2], BOND_LIST[int(order_idx)])

    for node, charge in formal_charges.items():
        if node in node_to_idx:
            rw_mol.GetAtomWithIdx(node_to_idx[node]).SetFormalCharge(int(charge))

    mol = rw_mol.GetMol()
    Chem.SanitizeMol(mol)
    return mol


def cap_terminal_dummy_atoms(mol: Chem.Mol) -> Chem.Mol:
    """
    功能目的：
        将聚合物首末端 dummy 原子原位替换为 H。

    输入参数：
        mol: 含首末端“*”的 RDKit Mol。

    返回值：
        封端后的 RDKit Mol，原子数量与顺序保持不变。

    可能报错或边界情况：
        dummy 原子数量不是 2 时抛出 ValueError。
    """
    dummy_indices = [atom.GetIdx() for atom in mol.GetAtoms() if atom.GetSymbol() == "*"]
    if len(dummy_indices) != 2:
        raise ValueError(f"封端前应有 2 个 dummy 原子，实际 {len(dummy_indices)}")
    rw_mol = RWMol(mol)
    for idx in dummy_indices:
        rw_mol.ReplaceAtom(idx, Chem.Atom(1))
        rw_mol.GetAtomWithIdx(idx).SetNoImplicit(False)
    rw_mol.UpdatePropertyCache(strict=False)
    capped = rw_mol.GetMol()
    Chem.SanitizeMol(capped)
    return capped


def _write_openbabel_structure_files(mol: Chem.Mol, prefixes: Sequence[Path]) -> None:
    """
    功能目的：
        使用 Open Babel 写出 mol2/pdb 结构文件，兼容后续 sobtop notebook。

    输入参数：
        mol: 已有 3D 构象并包含键级信息的 RDKit Mol。
        prefixes: 不带后缀的输出路径列表。

    返回值：
        无。

    关键流程：
        从 RDKit MolBlock 读入 Open Babel，分别写出 .mol2 和 .pdb。

    可能报错或边界情况：
        Open Babel 不可用时只打印警告，不影响 .xyz/.chg 的严格电荷验证。
    """
    try:
        from openbabel import openbabel as ob  # type: ignore
    except Exception as exc:
        print(f"[警告] Open Babel 不可用，跳过 mol2/pdb 写出: {exc}")
        return

    mol_block = Chem.MolToMolBlock(mol)
    for prefix in prefixes:
        prefix = Path(prefix)
        prefix.parent.mkdir(parents=True, exist_ok=True)
        ob_conversion = ob.OBConversion()
        ob_conversion.SetInAndOutFormats("sdf", "mol2")
        ob_mol = ob.OBMol()
        ok = ob_conversion.ReadString(ob_mol, mol_block)
        if not ok:
            print(f"[警告] Open Babel 读取 RDKit MolBlock 失败，跳过 {prefix}")
            continue
        ob_conversion.WriteFile(ob_mol, str(prefix.with_suffix(".mol2")))
        ob_conversion.SetOutFormat("pdb")
        ob_conversion.WriteFile(ob_mol, str(prefix.with_suffix(".pdb")))


def write_xyz(mol: Chem.Mol, xyz_path: Path, extra_structure_prefixes: Sequence[Path] | None = None) -> List[str]:
    """
    功能目的：
        生成 3D 构象并写出 XYZ 文件。

    输入参数：
        mol: 封端后的 RDKit Mol。
        xyz_path: 输出路径。
        extra_structure_prefixes: 额外写出 mol2/pdb/xyz 的不带后缀路径。

    返回值：
        XYZ 文件中的元素序列。

    可能报错或边界情况：
        构象生成失败时抛出 RuntimeError。
    """
    work_mol = Chem.Mol(mol)
    status = AllChem.EmbedMolecule(work_mol, useRandomCoords=True, randomSeed=20260428)
    if status != 0:
        raise RuntimeError("聚合物 3D 构象生成失败")
    try:
        AllChem.MMFFOptimizeMolecule(work_mol, mmffVariant="MMFF94")
    except Exception:
        AllChem.UFFOptimizeMolecule(work_mol)

    xyz_path.parent.mkdir(parents=True, exist_ok=True)
    xyz_text = Chem.MolToXYZBlock(work_mol)
    xyz_path.write_text(xyz_text, encoding="utf-8")
    if extra_structure_prefixes:
        for prefix in extra_structure_prefixes:
            prefix = Path(prefix)
            prefix.parent.mkdir(parents=True, exist_ok=True)
            prefix.with_suffix(".xyz").write_text(xyz_text, encoding="utf-8")
        _write_openbabel_structure_files(work_mol, extra_structure_prefixes)
    elements = [line.split()[0] for line in xyz_text.splitlines()[2:] if line.split()]
    return elements


def write_chg(
    xyz_path: Path,
    charges: Sequence[float],
    chg_path: Path,
    extra_chg_paths: Sequence[Path] | None = None,
) -> None:
    """
    功能目的：
        按 XYZ 原子顺序写出 .chg 文件。

    输入参数：
        xyz_path: 已生成的 XYZ 文件。
        charges: 与 XYZ 坐标行一一对应的电荷。
        chg_path: 输出 .chg 路径。
        extra_chg_paths: 额外写出的 .chg 路径。

    返回值：
        无。

    可能报错或边界情况：
        原子数与电荷数不一致时抛出 ValueError。
    """
    lines = xyz_path.read_text(encoding="utf-8").splitlines()
    coord_lines = [line for line in lines[2:] if line.split()]
    if len(coord_lines) != len(charges):
        raise ValueError(f"XYZ 原子数 {len(coord_lines)} 与电荷数 {len(charges)} 不一致")

    chg_path.parent.mkdir(parents=True, exist_ok=True)
    with chg_path.open("w", encoding="utf-8", newline="\n") as handle:
        for line, charge in zip(coord_lines, charges):
            parts = line.split()
            handle.write(
                f"{parts[0]} {float(parts[1]): .6f} {float(parts[2]): .6f} "
                f"{float(parts[3]): .6f} {float(charge): .10f}\n"
            )
    if extra_chg_paths:
        for extra_path in extra_chg_paths:
            extra_path = Path(extra_path)
            extra_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(chg_path, extra_path)


def build_random_copolymer(
    smiles_by_name: Dict[str, str],
    repeating_unit: Dict[str, int],
    mapped_charge_paths: Dict[str, Path],
    copolymer_name: str,
    base_dir: Path,
    random_seed: int = 2,
) -> PolymerBuildResult:
    """
    功能目的：
        端到端生成随机共聚物 XYZ 和 CHG。

    输入参数：
        smiles_by_name: 重复单元名称 -> SMILES。
        repeating_unit: 重复单元名称 -> 出现次数。
        mapped_charge_paths: 重复单元名称 -> mapped_charges.json。
        copolymer_name: 共聚物名称。
        base_dir: 任务根目录。
        random_seed: 随机种子。

    返回值：
        PolymerBuildResult。

    关键流程：
        先写 XYZ，再在元素顺序和总电荷校验通过后写 CHG，避免生成错误电荷文件。
    """
    base_dir = Path(base_dir)
    sequence = build_sequence(repeating_unit, random_seed)
    return build_polymer_from_sequence(
        smiles_by_name=smiles_by_name,
        sequence=sequence,
        mapped_charge_paths=mapped_charge_paths,
        polymer_name=copolymer_name,
        base_dir=base_dir,
        cyclic=False,
        repeating_unit_for_expected_charge=repeating_unit,
    )


def build_polymer_from_sequence(
    smiles_by_name: Dict[str, str],
    sequence: Sequence[str],
    mapped_charge_paths: Dict[str, Path],
    polymer_name: str,
    base_dir: Path,
    cyclic: bool = False,
    repeating_unit_for_expected_charge: Dict[str, int] | None = None,
) -> PolymerBuildResult:
    """
    功能目的：
        根据显式重复单元序列生成线性或环状聚合物 XYZ 和 CHG。

    输入参数：
        smiles_by_name: 重复单元名称 -> SMILES。
        sequence: 已确定的聚合顺序。
        mapped_charge_paths: 重复单元名称 -> mapped_charges.json。
        polymer_name: 输出聚合物名称。
        base_dir: 输出根目录。
        cyclic: 是否生成环状聚合物。
        repeating_unit_for_expected_charge: 可选的理论计数字典；不传时从 sequence 统计。

    返回值：
        PolymerBuildResult。

    关键流程：
        该函数是所有模式的共享核心。它只信任 mapped_charges.json，不再按
        RDKit 当前 atom_idx 直接读取 .chg 行，从根源避免 Gaussian/Multiwfn 顺序
        与含 dummy 重复单元顺序混用。

    可能报错或边界情况：
        元素顺序、原子数、总电荷任一校验失败时会中止，不写出错误 .chg。
    """
    if not sequence:
        raise ValueError("聚合物序列为空")

    base_dir = Path(base_dir)
    mapped_payloads = {
        name: load_mapped_charges(Path(path))
        for name, path in mapped_charge_paths.items()
    }
    missing = sorted(set(sequence).difference(smiles_by_name))
    if missing:
        raise KeyError(f"sequence 中存在 smiles_by_name 缺失的重复单元: {missing}")
    missing_mapped = sorted(set(sequence).difference(mapped_payloads))
    if missing_mapped:
        raise KeyError(f"sequence 中存在 mapped_charge_paths 缺失的重复单元: {missing_mapped}")

    if repeating_unit_for_expected_charge is None:
        repeating_unit_for_expected_charge = {
            name: int(sequence.count(name))
            for name in dict.fromkeys(sequence)
        }

    charge_dict, formal_charges, _, bonds_with_order = collect_polymer_graph(
        smiles_by_name=smiles_by_name,
        mapped_payloads=mapped_payloads,
        sequence=sequence,
        cyclic=bool(cyclic),
    )

    expected_charge = sum(
        int(repeating_unit_for_expected_charge[name]) * float(mapped_payloads[name]["formal_charge"])
        for name in repeating_unit_for_expected_charge
    )
    if not cyclic:
        # 中文注释：
        # 线性共聚物的首末端 H 来自“重复单元 RESP 封端模型”中的 dummy 替换 H。
        # 当链首和链尾属于不同重复单元时，这两个 H 的 RESP 电荷通常不会天然相加为 0；
        # 但最终聚合物业务语义要求总电荷等于各 SMILES* 重复单元形式电荷之和。
        # 因此只把差值平均补偿到两个人工端基 H 上，非 dummy 重复单元电荷保持 RESP 约束值不变。
        terminal_dummy_nodes = [node for node in charge_dict if node.startswith("*")]
        if len(terminal_dummy_nodes) != 2:
            raise ValueError(f"线性聚合物应保留 2 个端基 dummy，实际为 {len(terminal_dummy_nodes)}")
        current_charge = float(np.sum([float(value) for value in charge_dict.values()]))
        terminal_delta = expected_charge - current_charge
        if abs(terminal_delta) > FLOAT_TOL:
            correction = terminal_delta / 2.0
            for node in terminal_dummy_nodes:
                charge_dict[node] = float(charge_dict[node]) + correction

    node_list = list(charge_dict.keys())
    charge_list = [float(charge_dict[node]) for node in node_list]
    expected_elements = [capped_node_element(node) for node in node_list]

    mol = graph_to_mol(node_list, bonds_with_order, formal_charges)
    capped_mol = mol if cyclic else cap_terminal_dummy_atoms(mol)

    safe_polymer = safe_name(polymer_name)
    xyz_path = base_dir / "polymer_structure" / f"{safe_polymer}.xyz"
    chg_path = base_dir / "polymer_topology" / f"{safe_polymer}.chg"
    node_list_path = base_dir / "polymer_node_list.json"

    legacy_prefixes = [
        base_dir / safe_polymer,
        base_dir / f"polymer_{safe_polymer}",
        base_dir / "polymer_structure" / safe_polymer,
        base_dir / "polymer_structure" / f"polymer_{safe_polymer}",
    ]
    xyz_elements = write_xyz(capped_mol, xyz_path, extra_structure_prefixes=legacy_prefixes)
    if xyz_elements != expected_elements:
        mismatch = [
            (idx + 1, got, expected)
            for idx, (got, expected) in enumerate(zip(xyz_elements, expected_elements))
            if got != expected
        ][:10]
        raise ValueError(f"XYZ 元素序列与 node_list 不一致，前 10 个差异: {mismatch}")

    actual_charge = float(np.sum(charge_list))
    if abs(actual_charge - expected_charge) > FLOAT_TOL:
        raise ValueError(
            f"聚合物总电荷 {actual_charge:.8f} 与理论值 {expected_charge:.8f} 不一致"
        )

    node_list_path.write_text(json.dumps(node_list, ensure_ascii=False, indent=2), encoding="utf-8")
    write_chg(
        xyz_path,
        charge_list,
        chg_path,
        extra_chg_paths=[
            base_dir / f"{safe_polymer}.chg",
            base_dir / f"polymer_{safe_polymer}.chg",
            base_dir / "polymer_topology" / f"polymer_{safe_polymer}.chg",
        ],
    )

    return PolymerBuildResult(
        copolymer_name=polymer_name,
        cyclic=bool(cyclic),
        sequence=list(sequence),
        expected_charge=expected_charge,
        actual_charge=actual_charge,
        node_list=node_list,
        xyz_elements=xyz_elements,
        charge_list=charge_list,
        xyz_path=xyz_path,
        chg_path=chg_path,
        node_list_path=node_list_path,
    )


def build_block_copolymer(
    smiles_by_name: Dict[str, str],
    repeating_unit: Dict[str, int],
    number_of_blocks: int,
    mapped_charge_paths: Dict[str, Path],
    copolymer_name: str,
    base_dir: Path,
) -> PolymerBuildResult:
    """
    功能目的：
        生成线性嵌段共聚物。
    """
    sequence = build_block_sequence(repeating_unit, number_of_blocks)
    expanded_counts = {name: int(count) * int(number_of_blocks) for name, count in repeating_unit.items()}
    return build_polymer_from_sequence(
        smiles_by_name=smiles_by_name,
        sequence=sequence,
        mapped_charge_paths=mapped_charge_paths,
        polymer_name=copolymer_name,
        base_dir=base_dir,
        cyclic=False,
        repeating_unit_for_expected_charge=expanded_counts,
    )


def build_cyclic_block_copolymer(
    smiles_by_name: Dict[str, str],
    repeating_unit: Dict[str, int],
    number_of_blocks: int,
    mapped_charge_paths: Dict[str, Path],
    copolymer_name: str,
    base_dir: Path,
) -> PolymerBuildResult:
    """
    功能目的：
        生成环状嵌段共聚物。
    """
    sequence = build_block_sequence(repeating_unit, number_of_blocks)
    expanded_counts = {name: int(count) * int(number_of_blocks) for name, count in repeating_unit.items()}
    return build_polymer_from_sequence(
        smiles_by_name=smiles_by_name,
        sequence=sequence,
        mapped_charge_paths=mapped_charge_paths,
        polymer_name=copolymer_name,
        base_dir=base_dir,
        cyclic=True,
        repeating_unit_for_expected_charge=expanded_counts,
    )


def build_cyclic_random_copolymer(
    smiles_by_name: Dict[str, str],
    repeating_unit: Dict[str, int],
    mapped_charge_paths: Dict[str, Path],
    copolymer_name: str,
    base_dir: Path,
    random_seed: int = 2,
) -> PolymerBuildResult:
    """
    功能目的：
        生成环状随机共聚物。
    """
    sequence = build_sequence(repeating_unit, random_seed)
    return build_polymer_from_sequence(
        smiles_by_name=smiles_by_name,
        sequence=sequence,
        mapped_charge_paths=mapped_charge_paths,
        polymer_name=copolymer_name,
        base_dir=base_dir,
        cyclic=True,
        repeating_unit_for_expected_charge=repeating_unit,
    )


def build_homopolymer(
    smiles: str,
    repeating_unit: int,
    mapped_charge_path: Path,
    polymer_name: str,
    base_dir: Path,
) -> PolymerBuildResult:
    """
    功能目的：
        生成线性均聚物。
    """
    sequence = build_homopolymer_sequence(polymer_name, repeating_unit)
    return build_polymer_from_sequence(
        smiles_by_name={polymer_name: smiles},
        sequence=sequence,
        mapped_charge_paths={polymer_name: Path(mapped_charge_path)},
        polymer_name=polymer_name,
        base_dir=base_dir,
        cyclic=False,
        repeating_unit_for_expected_charge={polymer_name: int(repeating_unit)},
    )


def build_cyclic_homopolymer(
    smiles: str,
    repeating_unit: int,
    mapped_charge_path: Path,
    polymer_name: str,
    base_dir: Path,
) -> PolymerBuildResult:
    """
    功能目的：
        生成环状均聚物。

    输入参数：
        smiles: 含两个“*”聚合位点的重复单元 SMILES。
        repeating_unit: 聚合度。
        mapped_charge_path: 该重复单元的 mapped_charges.json。
        polymer_name: 输出聚合物名称。
        base_dir: 输出根目录。

    返回值：
        PolymerBuildResult。

    关键流程：
        使用均聚物序列，但在 build_polymer_from_sequence 中设置 cyclic=True。
        因此所有 dummy 节点都会被删除，首末聚合位点相邻真实原子直接闭环，
        最终 .chg 的总电荷等于 repeating_unit * SMILES* 形式电荷。

    可能报错或边界情况：
        mapped charge 缺失、闭环结构非法或电荷/元素校验失败时抛出异常。
    """
    sequence = build_homopolymer_sequence(polymer_name, repeating_unit)
    return build_polymer_from_sequence(
        smiles_by_name={polymer_name: smiles},
        sequence=sequence,
        mapped_charge_paths={polymer_name: Path(mapped_charge_path)},
        polymer_name=polymer_name,
        base_dir=base_dir,
        cyclic=True,
        repeating_unit_for_expected_charge={polymer_name: int(repeating_unit)},
    )


In [ ]:
# -*- coding: utf-8 -*-
"""
聚合物电荷映射结果验证工具。

功能目的：
    验证重复单元 mapped charge 与最终聚合物 .chg 是否满足电荷守恒、
    原子数一致和元素顺序一致。

输入参数：
    validate_full_result(...) 接收任务目录、重复单元信息、mapped charge 路径、
    node_list、XYZ 和 CHG 文件。

返回值：
    验证报告字典，并可写出 validation_report.json。

关键流程：
    1. 检查每个重复单元非 dummy 电荷和；
    2. 检查 dummy 节点映射到 Gaussian H；
    3. 检查最终 XYZ/CHG 原子数和元素序列；
    4. 检查最终总电荷等于理论总电荷。

可能报错或边界情况：
    任一校验失败会抛出 ValueError。
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List


FLOAT_TOL = 1.0e-4


def read_chg(chg_path: Path) -> List[Dict[str, Any]]:
    """
    功能目的：
        读取 .chg 文件。

    输入参数：
        chg_path: .chg 文件路径。

    返回值：
        原子记录列表。
    """
    rows: List[Dict[str, Any]] = []
    with Path(chg_path).open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, 1):
            parts = line.split()
            if not parts:
                continue
            if len(parts) < 5:
                raise ValueError(f"{chg_path} 第 {line_no} 行格式不完整")
            rows.append(
                {
                    "index1": len(rows) + 1,
                    "element": parts[0],
                    "charge": float(parts[-1]),
                }
            )
    return rows


def read_xyz_elements(xyz_path: Path) -> List[str]:
    """
    功能目的：
        读取 XYZ 坐标文件中的元素序列。
    """
    lines = Path(xyz_path).read_text(encoding="utf-8").splitlines()
    return [line.split()[0] for line in lines[2:] if line.split()]


def load_json(path: Path) -> Any:
    """
    功能目的：
        读取 UTF-8 JSON 文件。
    """
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def node_to_final_element(node: str) -> str:
    """
    功能目的：
        把 node_list 节点转换为最终封端后元素符号。
    """
    if node.startswith("*"):
        return "H"
    # 节点格式为 O1_CMC_4 或 C13_celllulose_7。
    chars = []
    for ch in node:
        if ch.isdigit() or ch == "_":
            break
        chars.append(ch)
    if not chars:
        raise ValueError(f"无法解析节点元素: {node}")
    return "".join(chars)


def validate_repeat_unit_payload(name: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    """
    功能目的：
        验证单个重复单元 mapped_charges。

    输入参数：
        name: 重复单元名称。
        payload: mapped_charges JSON 字典。

    返回值：
        重复单元验证摘要。

    关键流程：
        非 dummy 电荷和必须等于形式电荷；dummy 替换位点的 Gaussian 元素必须是 H。
    """
    mapped = payload["mapped_charges"]
    formal_charge = float(payload["formal_charge"])
    non_dummy_sum = 0.0
    dummy_nodes = []
    for node, entry in mapped.items():
        charge = float(entry["charge"])
        if entry["is_dummy_replacement"]:
            dummy_nodes.append(
                {
                    "node": node,
                    "gaussian_index1": int(entry["gaussian_index1"]),
                    "gaussian_symbol": entry["gaussian_symbol"],
                    "charge": charge,
                }
            )
            if entry["gaussian_symbol"] != "H":
                raise ValueError(f"{name}: dummy 节点 {node} 没有映射到 Gaussian H")
        else:
            non_dummy_sum += charge

    if len(dummy_nodes) != 2:
        raise ValueError(f"{name}: dummy 替换节点数量应为 2，实际 {len(dummy_nodes)}")
    if abs(non_dummy_sum - formal_charge) > FLOAT_TOL:
        raise ValueError(
            f"{name}: 非 dummy 电荷和 {non_dummy_sum:.8f} 与形式电荷 {formal_charge:.8f} 不一致"
        )
    return {
        "name": name,
        "formal_charge": formal_charge,
        "non_dummy_charge_sum": non_dummy_sum,
        "dummy_nodes": dummy_nodes,
    }


def validate_full_result(
    base_dir: Path,
    repeating_unit: Dict[str, int],
    mapped_charge_paths: Dict[str, Path],
    node_list_path: Path,
    xyz_path: Path,
    chg_path: Path,
    report_path: Path | None = None,
    old_reference_charge: float | None = 4.4005007164,
) -> Dict[str, Any]:
    """
    功能目的：
        完整验证聚合物电荷映射修复结果。

    输入参数：
        base_dir: 任务根目录。
        repeating_unit: 名称 -> 重复次数。
        mapped_charge_paths: 名称 -> mapped_charges.json。
        node_list_path: 聚合物节点列表。
        xyz_path/chg_path: 最终结构和电荷文件。
        report_path: 可选报告输出路径。
        old_reference_charge: 旧错误结果的总电荷，仅用于对照记录。

    返回值：
        验证报告字典。

    可能报错或边界情况：
        任一校验失败会抛出 ValueError。
    """
    base_dir = Path(base_dir)
    unit_reports = {}
    expected_charge = 0.0
    for name, count in repeating_unit.items():
        payload = load_json(Path(mapped_charge_paths[name]))
        unit_reports[name] = validate_repeat_unit_payload(name, payload)
        expected_charge += int(count) * float(payload["formal_charge"])

    node_list = load_json(Path(node_list_path))
    xyz_elements = read_xyz_elements(Path(xyz_path))
    chg_rows = read_chg(Path(chg_path))
    chg_elements = [row["element"] for row in chg_rows]
    node_elements = [node_to_final_element(node) for node in node_list]

    if len(node_list) != len(xyz_elements):
        raise ValueError(f"node_list 长度 {len(node_list)} 与 XYZ 原子数 {len(xyz_elements)} 不一致")
    if len(chg_rows) != len(xyz_elements):
        raise ValueError(f"CHG 原子数 {len(chg_rows)} 与 XYZ 原子数 {len(xyz_elements)} 不一致")
    if node_elements != xyz_elements:
        raise ValueError("node_list 元素序列与 XYZ 元素序列不一致")
    if chg_elements != xyz_elements:
        raise ValueError("CHG 元素序列与 XYZ 元素序列不一致")

    actual_charge = sum(float(row["charge"]) for row in chg_rows)
    if abs(actual_charge - expected_charge) > FLOAT_TOL:
        raise ValueError(
            f"聚合物总电荷 {actual_charge:.8f} 与理论值 {expected_charge:.8f} 不一致"
        )

    report = {
        "base_dir": str(base_dir.resolve()),
        "status": "pass",
        "expected_polymer_charge": expected_charge,
        "actual_polymer_charge": actual_charge,
        "old_reference_charge": old_reference_charge,
        "atom_count": len(xyz_elements),
        "unit_reports": unit_reports,
        "files": {
            "node_list": str(Path(node_list_path)),
            "xyz": str(Path(xyz_path)),
            "chg": str(Path(chg_path)),
        },
    }
    if report_path is not None:
        Path(report_path).write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return report



In [ ]:
# 固定参数：按需修改
EXCEL_FILE = "System_random_copolymer.xlsx"
POLYMER_MODE = "cyclic-random"  # homopolymer / cyclic-homopolymer / random / cyclic-random / block / cyclic-block
RANDOM_SEED = 2

import json
from pathlib import Path
from typing import Dict

import pandas as pd


def read_polymer_rows(excel_path: Path) -> pd.DataFrame:
    """
    功能目的：读取体系 Excel 并筛选真正需要处理的聚合物重复单元行。
    输入参数：excel_path，体系 Excel 文件。
    返回值：聚合物重复单元 DataFrame。
    关键流程：先去掉 Name/SMILES/repeating unit 任一为空的占位行，再按 is polymer 列筛选。
    可能报错或边界情况：缺列、空输入、重复名称或 repeating unit 非法时抛出异常。
    """
    df = pd.read_excel(excel_path)
    missing = {"Name", "SMILES", "repeating unit"}.difference(df.columns)
    if missing:
        raise ValueError(f"{excel_path} 缺少必要列: {sorted(missing)}")

    work_df = df.dropna(subset=["Name", "SMILES", "repeating unit"]).copy()
    work_df["Name"] = work_df["Name"].astype(str).str.strip()
    work_df["SMILES"] = work_df["SMILES"].astype(str).str.strip()
    work_df = work_df[
        (work_df["Name"] != "")
        & (work_df["SMILES"] != "")
        & (work_df["Name"].str.lower() != "nan")
        & (work_df["SMILES"].str.lower() != "nan")
    ].copy()

    if "is polymer" in work_df.columns:
        polymer_df = work_df[work_df["is polymer"].astype(bool)].copy()
    else:
        polymer_df = work_df.copy()

    if polymer_df.empty:
        raise ValueError(f"{excel_path} 没有有效聚合物行")
    if polymer_df["Name"].astype(str).duplicated().any():
        duplicated = polymer_df.loc[polymer_df["Name"].astype(str).duplicated(), "Name"].astype(str).tolist()
        raise ValueError(f"重复单元名称重复: {duplicated}")
    return polymer_df


def get_copolymer_name(polymer_df: pd.DataFrame, default_name: str | None = None) -> str:
    """
    功能目的：读取共聚物名称；旧均聚物表格没有 copolymer_name 时使用默认名称。
    输入参数：polymer_df 与 default_name。
    返回值：聚合物输出名称。
    可能报错或边界情况：共聚物模式缺少 copolymer_name 时抛出异常。
    """
    if "copolymer_name" in polymer_df.columns:
        value = polymer_df["copolymer_name"].iloc[0]
        if not pd.isna(value) and str(value).strip() and str(value).lower() != "nan":
            return str(value)
    if default_name is not None:
        return default_name
    raise ValueError("共聚物模式需要 copolymer_name 列")


def build_input_dicts(polymer_df: pd.DataFrame):
    """
    功能目的：从 DataFrame 生成 SMILES、重复次数和 mapped charge 路径字典。
    输入参数：聚合物行 DataFrame。
    返回值：(smiles_by_name, repeating_unit, mapped_charge_paths)。
    """
    smiles_by_name = {str(row["Name"]): str(row["SMILES"]) for _, row in polymer_df.iterrows()}
    repeating_unit = {str(row["Name"]): int(row["repeating unit"]) for _, row in polymer_df.iterrows()}
    mapped_charge_paths = {name: Path("mapped_charges") / f"{safe_name(name)}_mapped_charges.json" for name in smiles_by_name}
    missing = [str(path) for path in mapped_charge_paths.values() if not Path(path).exists()]
    if missing:
        raise FileNotFoundError("缺少 mapped_charges，请先运行 1_Polymer_RESP_repeat_unit.ipynb: " + ", ".join(missing))
    return smiles_by_name, repeating_unit, mapped_charge_paths


def save_repeat_unit_atom_lists(mapped_charge_paths: Dict[str, Path]) -> None:
    """
    功能目的：保存 {name}_atom_list.json，供后续 itp 参数替换 notebook 使用。
    输入参数：名称到 mapped_charges.json 的路径字典。
    返回值：无。
    关键流程：直接使用 mapped_charges 的节点顺序，避免重新用 RDKit/canonical SMILES 推断原子顺序。
    """
    for name, path in mapped_charge_paths.items():
        payload = load_mapped_charges(Path(path))
        atom_list = list(payload["mapped_charges"].keys())
        out_path = Path(f"{safe_name(name)}_atom_list.json")
        out_path.write_text(json.dumps(atom_list, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"[OK] 写出重复单元 atom_list: {out_path}")


base_dir = Path.cwd()
polymer_df = read_polymer_rows(Path(EXCEL_FILE))
smiles_by_name, repeating_unit, mapped_charge_paths = build_input_dicts(polymer_df)
save_repeat_unit_atom_lists(mapped_charge_paths)

if POLYMER_MODE == "homopolymer":
    reports = []
    for _, row in polymer_df.iterrows():
        name = str(row["Name"])
        result = build_homopolymer(
            smiles=smiles_by_name[name],
            repeating_unit=repeating_unit[name],
            mapped_charge_path=mapped_charge_paths[name],
            polymer_name=name,
            base_dir=base_dir,
        )
        report = validate_full_result(
            base_dir=base_dir,
            repeating_unit={name: repeating_unit[name]},
            mapped_charge_paths={name: mapped_charge_paths[name]},
            node_list_path=result.node_list_path,
            xyz_path=result.xyz_path,
            chg_path=result.chg_path,
            report_path=base_dir / f"validation_report_{safe_name(name)}.json",
            old_reference_charge=None,
        )
        reports.append(report)
    print(json.dumps(reports, ensure_ascii=False, indent=2))

elif POLYMER_MODE == "cyclic-homopolymer":
    reports = []
    for _, row in polymer_df.iterrows():
        name = str(row["Name"])
        result = build_cyclic_homopolymer(
            smiles=smiles_by_name[name],
            repeating_unit=repeating_unit[name],
            mapped_charge_path=mapped_charge_paths[name],
            polymer_name=name,
            base_dir=base_dir,
        )
        report = validate_full_result(
            base_dir=base_dir,
            repeating_unit={name: repeating_unit[name]},
            mapped_charge_paths={name: mapped_charge_paths[name]},
            node_list_path=result.node_list_path,
            xyz_path=result.xyz_path,
            chg_path=result.chg_path,
            report_path=base_dir / f"validation_report_{safe_name(name)}.json",
            old_reference_charge=None,
        )
        reports.append(report)
    print(json.dumps(reports, ensure_ascii=False, indent=2))

elif POLYMER_MODE == "random":
    copolymer_name = get_copolymer_name(polymer_df)
    result = build_random_copolymer(
        smiles_by_name=smiles_by_name,
        repeating_unit=repeating_unit,
        mapped_charge_paths=mapped_charge_paths,
        copolymer_name=copolymer_name,
        base_dir=base_dir,
        random_seed=RANDOM_SEED,
    )
    report = validate_full_result(
        base_dir=base_dir,
        repeating_unit=repeating_unit,
        mapped_charge_paths=mapped_charge_paths,
        node_list_path=result.node_list_path,
        xyz_path=result.xyz_path,
        chg_path=result.chg_path,
        report_path=base_dir / "validation_report.json",
        old_reference_charge=None,
    )
    print(json.dumps(report, ensure_ascii=False, indent=2))

elif POLYMER_MODE == "cyclic-random":
    copolymer_name = get_copolymer_name(polymer_df)
    result = build_cyclic_random_copolymer(
        smiles_by_name=smiles_by_name,
        repeating_unit=repeating_unit,
        mapped_charge_paths=mapped_charge_paths,
        copolymer_name=copolymer_name,
        base_dir=base_dir,
        random_seed=RANDOM_SEED,
    )
    report = validate_full_result(
        base_dir=base_dir,
        repeating_unit=repeating_unit,
        mapped_charge_paths=mapped_charge_paths,
        node_list_path=result.node_list_path,
        xyz_path=result.xyz_path,
        chg_path=result.chg_path,
        report_path=base_dir / "validation_report.json",
        old_reference_charge=None,
    )
    print(json.dumps(report, ensure_ascii=False, indent=2))

elif POLYMER_MODE == "block":
    copolymer_name = get_copolymer_name(polymer_df)
    if "Number of blocks" not in polymer_df.columns:
        raise ValueError("block 模式需要 Number of blocks 列")
    number_of_blocks = int(polymer_df["Number of blocks"].iloc[0])
    result = build_block_copolymer(
        smiles_by_name=smiles_by_name,
        repeating_unit=repeating_unit,
        number_of_blocks=number_of_blocks,
        mapped_charge_paths=mapped_charge_paths,
        copolymer_name=copolymer_name,
        base_dir=base_dir,
    )
    validation_counts = {name: count * number_of_blocks for name, count in repeating_unit.items()}
    report = validate_full_result(
        base_dir=base_dir,
        repeating_unit=validation_counts,
        mapped_charge_paths=mapped_charge_paths,
        node_list_path=result.node_list_path,
        xyz_path=result.xyz_path,
        chg_path=result.chg_path,
        report_path=base_dir / "validation_report.json",
        old_reference_charge=None,
    )
    print(json.dumps(report, ensure_ascii=False, indent=2))

elif POLYMER_MODE == "cyclic-block":
    copolymer_name = get_copolymer_name(polymer_df)
    if "Number of blocks" not in polymer_df.columns:
        raise ValueError("cyclic-block 模式需要 Number of blocks 列")
    number_of_blocks = int(polymer_df["Number of blocks"].iloc[0])
    result = build_cyclic_block_copolymer(
        smiles_by_name=smiles_by_name,
        repeating_unit=repeating_unit,
        number_of_blocks=number_of_blocks,
        mapped_charge_paths=mapped_charge_paths,
        copolymer_name=copolymer_name,
        base_dir=base_dir,
    )
    validation_counts = {name: count * number_of_blocks for name, count in repeating_unit.items()}
    report = validate_full_result(
        base_dir=base_dir,
        repeating_unit=validation_counts,
        mapped_charge_paths=mapped_charge_paths,
        node_list_path=result.node_list_path,
        xyz_path=result.xyz_path,
        chg_path=result.chg_path,
        report_path=base_dir / "validation_report.json",
        old_reference_charge=None,
    )
    print(json.dumps(report, ensure_ascii=False, indent=2))

else:
    raise ValueError(f"未知 POLYMER_MODE: {POLYMER_MODE}")
